NER - NAMED ENTITY RECOGNITION MODEL TUNING

In [1]:
import pandas as pd

file_path = 'helper_docs/ner_examples.csv'

data = pd.read_csv(file_path)
data.head(10)

,sentence_id,words,labels
0,0,I,0
1,0,need,0
2,0,a,0
3,0,loan,0
4,0,of,0
5,0,50000,B-AMOUNT
6,0,dollars,0
7,0,for,0
8,0,12,B-TERM
9,0,months,0


In [2]:
# reconstruct the sentences and their corresponding labels

grouped_data = data.groupby("sentence_id").agg(lambda x: list(x))
grouped_data.head()


,words,labels
sentence_id,,
0,"[I, need, a, loan, of, 50000, dollars, for, 12...","[0, 0, 0, 0, 0, B-AMOUNT, 0, 0, B-TERM, 0]"
1,"[Can, I, get, a, loan, for, 150000, euros, for...","[0, 0, 0, 0, 0, 0, B-AMOUNT, 0, 0, 0, 0, 0, B-..."
2,"[Looking, for, a, 200000, euro, loan, for, 36,...","[0, 0, 0, B-AMOUNT, 0, 0, 0, B-TERM, 0]"
3,"[I, require, a, loan, amounting, to, 250000, f...","[0, 0, 0, 0, 0, 0, B-AMOUNT, 0, 0, B-TERM, 0, 0]"
4,"[Seeking, a, 30000, loan, for, a, term, of, 60...","[0, 0, B-AMOUNT, 0, 0, 0, 0, 0, B-TERM, 0]"


In [3]:
from transformers import BertTokenizerFast, BertForTokenClassification, AdamW
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

max_len = 128  
label_list = list(set(data['labels'].unique())) 
label_map = {label: i for i, label in enumerate(label_list)}
label_map

{'0': 0, 'B-AMOUNT': 1, 'B-TERM': 2}

In [5]:
import pickle
with open('helper_docs/ner_label_list.pkl', 'wb') as file:
    pickle.dump(label_list, file)

In [6]:

class NERDataset(Dataset):
    def __init__(self, grouped_data, tokenizer, max_len):
        self.sentences = grouped_data['words'].tolist()
        self.labels = grouped_data['labels'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(sentence, is_split_into_words=True, return_offsets_mapping=True, padding='max_length', truncation=True, max_length=self.max_len)
        
        # Convert labels to label IDs
        labels = [label_map[l] for l in label]
        encoded_labels = np.ones(len(encoding['offset_mapping']), dtype=int) * -100
        i = 0
        for idx, mapping in enumerate(encoding['offset_mapping']):
            if mapping[0] == 0 and i < len(labels):
                encoded_labels[idx] = labels[i]
                i += 1

        item = {key: torch.as_tensor(val) for key, val in encoding.items()}
        item['labels'] = torch.as_tensor(encoded_labels)
        return item


In [7]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(grouped_data, test_size=0.1)
train_dataset = NERDataset(train_data, tokenizer, max_len)
val_dataset = NERDataset(val_data, tokenizer, max_len)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [9]:
model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=len(label_list))

optimizer = AdamW(model.parameters(), lr=5e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/okanyenigun/Desktop/codes/projects/genai_work/notebooks/venv/lib/python3.11/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [10]:
for epoch in range(30):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.414484441280365
Epoch 2, Loss: 0.15565060079097748
Epoch 3, Loss: 0.04048892855644226
Epoch 4, Loss: 0.00726706488057971
Epoch 5, Loss: 0.003159907180815935
Epoch 6, Loss: 0.001720292610116303
Epoch 7, Loss: 0.001196367316879332
Epoch 8, Loss: 0.0010930767748504877
Epoch 9, Loss: 0.0009869846981018782
Epoch 10, Loss: 0.0007951856823638082
Epoch 11, Loss: 0.0007025377708487213
Epoch 12, Loss: 0.0006272531463764608
Epoch 13, Loss: 0.00059355708071962
Epoch 14, Loss: 0.0005405375850386918
Epoch 15, Loss: 0.000557278806809336
Epoch 16, Loss: 0.0005543993320316076
Epoch 17, Loss: 0.00047973651089705527
Epoch 18, Loss: 0.0004666117310989648
Epoch 19, Loss: 0.00042460300028324127
Epoch 20, Loss: 0.0003651720762718469
Epoch 21, Loss: 0.0003375532105565071
Epoch 22, Loss: 0.00036288637784309685
Epoch 23, Loss: 0.00033542438177391887
Epoch 24, Loss: 0.0003206448454875499
Epoch 25, Loss: 0.00034327455796301365
Epoch 26, Loss: 0.0003255315823480487
Epoch 27, Loss: 0.00027890427736

In [11]:
# Evaluation

model.eval()
predictions, true_labels = [], []
for batch in val_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    predictions.extend([list(p) for p in np.argmax(logits.detach().cpu().numpy(), axis=2)])
    true_labels.extend(batch['labels'].numpy())

In [12]:
from sklearn.metrics import classification_report

# predictions to label names

pred_tags = [[label_list[p_i] for p_i in p if p_i != -100] for p in predictions]
true_tags = [[label_list[l_i] for l_i in l if l_i != -100] for l in true_labels]


In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit([label_list])
true_label_ids = mlb.transform(true_tags)
pred_label_ids = mlb.transform(pred_tags)

print(classification_report(true_label_ids, pred_label_ids, target_names=mlb.classes_))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
    B-AMOUNT       1.00      1.00      1.00        11
      B-TERM       1.00      1.00      1.00        11

   micro avg       1.00      1.00      1.00        33
   macro avg       1.00      1.00      1.00        33
weighted avg       1.00      1.00      1.00        33
 samples avg       1.00      1.00      1.00        33



In [14]:
# save
model_path = "ner_trained_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)


('ner_trained_model/tokenizer_config.json',
 'ner_trained_model/special_tokens_map.json',
 'ner_trained_model/vocab.txt',
 'ner_trained_model/added_tokens.json',
 'ner_trained_model/tokenizer.json')

In [15]:
# load

model_path = "ner_trained_model"
model = BertForTokenClassification.from_pretrained(model_path)
tokenizer = BertTokenizerFast.from_pretrained(model_path)


In [16]:
# example

text = "I am looking for a 12 month loan for 150000 lira"

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits

# predictions to labels
predictions = torch.argmax(logits, dim=2)
predicted_label_ids = predictions[0].tolist()  # Assuming batch size is 1
predicted_labels = [label_list[label_id] for label_id in predicted_label_ids]

print(predicted_labels, len(predicted_labels))



['0', '0', '0', '0', '0', 'B-TERM', '0', '0', '0', 'B-AMOUNT', '0', '0', '0', '0', '0'] 15


In [17]:

# Extract specific entities
extracted_entities = {
    "B-AMOUNT": [],
    "B-TERM": []
}

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("tokens: ", tokens, len(tokens))
for token, label in zip(tokens, predicted_labels):
    if label == "B-AMOUNT":
        extracted_entities["B-AMOUNT"].append(token)
    elif label == "B-TERM":
        extracted_entities["B-TERM"].append(token)

print(extracted_entities)

tokens:  ['[CLS]', 'i', 'am', 'looking', 'for', 'a', '12', 'month', 'loan', 'for', '1500', '##00', 'li', '##ra', '[SEP]'] 15
{'B-AMOUNT': ['for'], 'B-TERM': ['a']}


In [18]:
# also check the adjacents

amount_idx = predicted_labels.index("B-AMOUNT")
text_list = text.split(" ")
loan_amount = text_list[amount_idx]
try:
    loan_amount = int(loan_amount)
except:
    try:
        loan_amount = int(tokens[amount_idx+1])
    except:
        loan_amount = 0
loan_amount

150000

In [19]:
term_idx = predicted_labels.index("B-TERM")
loan_term = tokens[term_idx]
try:
    loan_term = int(loan_term)
except:
    try:
        loan_term = int(tokens[term_idx+1])
    except:
        loan_amount = 0
loan_term

12